# Fine-tune a native Kinyarwanda voice (MMS-TTS) — free GPU

Runs on **Kaggle** (T4/P100, ~30h/week free) or **Google Colab** (free T4).
Base model: `facebook/mms-tts-kin` → fine-tuned single-speaker native voice.

**Before running:** enable GPU (Kaggle: Settings → Accelerator → GPU T4 x2 / Colab: Runtime → Change runtime type → T4 GPU).

**You need:** a HuggingFace token with *write* access, and your dataset (from `prepare_dataset.py`) either pushed as a private HF dataset or uploaded as a Kaggle dataset / zip.

## 1. Setup — clone the official tool + install deps

In [ ]:
!git clone https://github.com/ylacombe/finetune-hf-vits.git
%cd finetune-hf-vits
!pip install -q -r requirements.txt
# monotonic-align: VITS alignment, compiled once
%cd monotonic_align && !python setup.py build_ext --inplace && %cd ..
# Kinyarwanda is Latin-script -> MMS tokenizer is character-based -> NO uroman / espeak needed.

## 2. Authenticate to the HuggingFace Hub (for dataset pull + model push)

In [ ]:
from huggingface_hub import login
import os
# Kaggle: add HF_TOKEN as a Secret (Add-ons -> Secrets). Colab: use userdata / paste.
HF_TOKEN = os.environ.get('HF_TOKEN') or 'hf_xxx_PASTE_YOUR_WRITE_TOKEN'
login(HF_TOKEN)

## 3. Build the discriminator checkpoint
VITS fine-tuning is adversarial — the repo reconstructs a discriminator from the generator checkpoint.

In [ ]:
!python convert_original_discriminator_checkpoint.py \
    --language_code kin \
    --checkpoint_dir ./mms-tts-kin-train

## 4. Drop in the config
Edit `finetune_config.json` (from this repo's `tts-finetune/`) so:
- `hub_model_id` = your `username/mms-tts-kin-native`
- `dataset_name` = your private HF dataset id **or** a local path with `metadata.csv`
- `model_name_or_path` = `./mms-tts-kin-train` (the discriminator-augmented checkpoint from step 3)

Upload it next to the notebook, then:

In [ ]:
# Point model_name_or_path at the local discriminator checkpoint built in step 3:
import json, pathlib
cfg = json.loads(pathlib.Path('finetune_config.json').read_text())
cfg['model_name_or_path'] = './mms-tts-kin-train'
pathlib.Path('finetune_config.json').write_text(json.dumps(cfg, indent=2, ensure_ascii=False))
print(json.dumps(cfg, indent=2, ensure_ascii=False))

## 5. Train (≈ < 1h on T4 for 30-60 min of audio)

In [ ]:
!accelerate launch run_vits_finetuning.py ./finetune_config.json

## 6. Quick listen — A/B the fine-tuned voice vs. stock MMS

In [ ]:
from transformers import VitsModel, AutoTokenizer
import torch, scipy.io.wavfile, numpy as np
from IPython.display import Audio, display

TEXT = 'Muraho neza, murakaza neza kuri Tuganire. Iki ni ijwi nyarwanda.'

def synth(model_id, label):
    m = VitsModel.from_pretrained(model_id)
    tok = AutoTokenizer.from_pretrained(model_id)
    inp = tok(TEXT, return_tensors='pt')
    with torch.no_grad():
        wav = m(**inp).waveform[0].cpu().numpy()
    print(label)
    display(Audio(wav, rate=m.config.sampling_rate))

synth('facebook/mms-tts-kin', 'BEFORE (stock MMS)')
synth('./mms-tts-kin-native', 'AFTER (fine-tuned native)')

## 7. Deploy
If `push_to_hub: true`, the model is already on the Hub. In production set:
```
MMS_RW_MODEL=YOUR_HF_USERNAME/mms-tts-kin-native
```
and restart `tts-server` — `tts_server.py` reads this env var and loads your voice instead of the stock checkpoint. No code change, no API change.

**Not good enough yet?** Add more clean audio (or more epochs), retrain, re-listen. Iterate.